# 🧠 AI-Based Mental Health Sentiment Monitoring System
### Simple RNN Model — Google Colab Notebook

**Dataset:** [Sentiment Analysis for Mental Health](https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health)  
**Classes:** Normal, Depression, Suicidal, Anxiety, Bipolar, Stress, Personality Disorder

## 📦 Install & Import Dependencies

In [ ]:
!pip install tensorflow scikit-learn pandas numpy matplotlib seaborn nltk pickle5 -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import pickle
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {tf.config.list_physical_devices("GPU")}')

---
## Task 1 — Dataset Understanding

In [ ]:
# Load dataset
df = pd.read_csv('Combined Data.csv')
df = df[['statement', 'status']].dropna()
df.columns = ['text', 'label']

print('=== Dataset Overview ===')
print(f'Total samples     : {len(df)}')
print(f'Number of classes : {df["label"].nunique()}')
print(f'Classes           : {df["label"].unique().tolist()}')
print()
df.head()

In [ ]:
# Class distribution
class_counts = df['label'].value_counts()
print('=== Class Distribution ===')
print(class_counts)

plt.figure(figsize=(10, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')
plt.title('Class Distribution — Mental Health Sentiment', fontsize=14)
plt.xlabel('Sentiment Class')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Text length analysis
df['text_length'] = df['text'].apply(lambda x: len(str(x).split()))

print('=== Text Length Statistics ===')
print(df['text_length'].describe())
print(f'\nAverage sentence length : {df["text_length"].mean():.1f} words')
print(f'Max sentence length     : {df["text_length"].max()} words')
print(f'95th percentile         : {int(np.percentile(df["text_length"], 95))} words')

plt.figure(figsize=(10, 4))
plt.hist(df['text_length'], bins=50, color='steelblue', edgecolor='white')
plt.title('Distribution of Text Lengths')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.axvline(df['text_length'].mean(), color='red', linestyle='--', label=f'Mean: {df["text_length"].mean():.1f}')
plt.legend()
plt.tight_layout()
plt.savefig('text_length_distribution.png', dpi=150)
plt.show()

---
## Task 2 — Text Preprocessing

In [ ]:
STOPWORDS = set(stopwords.words('english'))

def preprocess_text(text):
    """Full preprocessing pipeline:
    1. Lowercase
    2. Remove punctuation
    3. Remove stopwords
    4. Strip extra whitespace
    """
    text = str(text).lower()                                      # 1. Lowercase
    text = re.sub(r'\d+', '', text)                               # Remove digits
    text = text.translate(str.maketrans('', '', string.punctuation))  # 2. Remove punctuation
    tokens = text.split()                                         # Tokenize
    tokens = [w for w in tokens if w not in STOPWORDS]           # 3. Remove stopwords
    return ' '.join(tokens)                                       # Rejoin

# Apply preprocessing
df['clean_text'] = df['text'].apply(preprocess_text)

# Show before/after
print('=== Preprocessing Examples ===')
for i in range(3):
    print(f'\nOriginal : {df["text"].iloc[i]}')
    print(f'Cleaned  : {df["clean_text"].iloc[i]}')

---
## Task 3 — Sequence Preparation

### Why RNN Cannot Directly Understand Raw Text

> RNNs are mathematical models that operate on **numerical tensors**, not strings.  
> Raw text must be converted to numbers through three steps:
> 1. **Tokenization** — split text into individual words (tokens)
> 2. **Word Indexing** — assign a unique integer ID to each word (vocabulary)
> 3. **Padding** — make all sequences the same length so they can be batched into a matrix
>
> The **Embedding layer** then maps each integer ID to a dense vector of floats,  
> which the RNN can process as meaningful numerical representations.

In [ ]:
# Hyperparameters
VOCAB_SIZE   = 20000   # Top N most frequent words
MAX_LEN      = 100     # Pad/truncate all sequences to this length
EMBED_DIM    = 64      # Embedding vector size
OOV_TOKEN    = '<OOV>' # Out-of-vocabulary token

# Tokenizer
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(df['clean_text'])

word_index = tokenizer.word_index
print(f'Vocabulary size (full)  : {len(word_index)}')
print(f'Vocabulary size (capped): {VOCAB_SIZE}')
print(f'Sample word→index       : {dict(list(word_index.items())[:10])}')

In [ ]:
# Convert texts to padded sequences
sequences = tokenizer.texts_to_sequences(df['clean_text'])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

print(f'Shape of X (padded sequences): {X.shape}')
print(f'\nExample raw text  : {df["clean_text"].iloc[1]}')
print(f'Example sequence  : {sequences[1]}')
print(f'Example padded    : {X[1]}')

In [ ]:
# Label encoding
le = LabelEncoder()
y_encoded = le.fit_transform(df['label'])
NUM_CLASSES = len(le.classes_)
y = to_categorical(y_encoded, num_classes=NUM_CLASSES)

print(f'Classes : {le.classes_.tolist()}')
print(f'y shape : {y.shape}')

# Train / Validation / Test split  (70 / 15 / 15)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y_encoded)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f'\nTrain size      : {X_train.shape[0]}')
print(f'Validation size : {X_val.shape[0]}')
print(f'Test size       : {X_test.shape[0]}')

---
## Task 4 — Build Simple RNN Architecture

In [ ]:
def build_rnn_model(vocab_size, embed_dim, max_len, num_classes):
    model = Sequential([
        # Layer 1: Embedding — maps integer word IDs → dense vectors
        Embedding(input_dim=vocab_size,
                  output_dim=embed_dim,
                  input_length=max_len,
                  name='embedding'),

        # Layer 2: SimpleRNN — processes sequence, maintains hidden state
        SimpleRNN(units=128,
                  activation='tanh',
                  return_sequences=False,
                  name='simple_rnn'),

        # Regularization
        Dropout(0.4, name='dropout'),

        # Layer 3: Dense hidden layer
        Dense(64, activation='relu', name='dense_hidden'),
        Dropout(0.3, name='dropout_2'),

        # Layer 4: Output layer — softmax for multi-class
        Dense(num_classes, activation='softmax', name='output')
    ])
    return model

model = build_rnn_model(VOCAB_SIZE, EMBED_DIM, MAX_LEN, NUM_CLASSES)
model.summary()

---
## Task 5 — Model Training

In [ ]:
# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

# Train
BATCH_SIZE = 64
EPOCHS     = 20

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'],     label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

---
## Task 6 — Model Evaluation

In [ ]:
# Predictions on test set
y_pred_probs = model.predict(X_test)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = np.argmax(y_test, axis=1)

# Metrics
print('=== Classification Report ===')
print(classification_report(y_true, y_pred, target_names=le.classes_))

acc = accuracy_score(y_true, y_pred)
print(f'Overall Test Accuracy: {acc:.4f}')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — Mental Health Sentiment RNN', fontsize=14)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

---
## Task 7 — Sequence Understanding

### How RNN Remembers Previous Words

A **Simple RNN** processes one word at a time, left to right.  
At each time step `t`, it computes a **hidden state** `h_t`:

```
h_t = tanh(W_h · h_{t-1}  +  W_x · x_t  +  b)
```

| Symbol | Meaning |
|--------|---------|
| `h_t`  | Hidden state at step t (memory of everything seen so far) |
| `h_{t-1}` | Previous hidden state (carries past context) |
| `x_t`  | Current word embedding |
| `W_h`, `W_x` | Learnable weight matrices |
| `tanh` | Non-linear activation |

**Sequential learning behavior:**  
- The hidden state acts as a *running summary* of the sentence so far  
- After processing all words, the final `h_T` captures the overall meaning  
- This final state is passed to the Dense layer for classification  
- **Limitation:** Simple RNNs suffer from vanishing gradients on long sequences (LSTM/GRU solve this)

In [ ]:
# Visualize hidden state concept
sample_sentence = "I feel very anxious and cannot sleep at night"
tokens = sample_sentence.split()

fig, ax = plt.subplots(figsize=(14, 3))
for i, word in enumerate(tokens):
    ax.annotate('', xy=(i+1, 0.5), xytext=(i, 0.5),
                arrowprops=dict(arrowstyle='->', color='steelblue', lw=2))
    ax.text(i, 0.5, word, ha='center', va='bottom', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue'))
    ax.text(i, 0.3, f'h_{i}', ha='center', va='top', fontsize=8, color='gray')

ax.text(len(tokens), 0.5, 'Output\n(Sentiment)', ha='center', va='center',
        fontsize=10, bbox=dict(boxstyle='round,pad=0.4', facecolor='lightgreen'))
ax.set_xlim(-0.5, len(tokens)+0.5)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('RNN Sequential Processing — Hidden State Flow', fontsize=12)
plt.tight_layout()
plt.savefig('rnn_hidden_state.png', dpi=150)
plt.show()

---
## Task 8 — Real-Time Prediction

In [ ]:
def predict_sentiment(text, model, tokenizer, le, max_len=MAX_LEN):
    """Predict sentiment for a single text input."""
    cleaned   = preprocess_text(text)
    sequence  = tokenizer.texts_to_sequences([cleaned])
    padded    = pad_sequences(sequence, maxlen=max_len, padding='post', truncating='post')
    probs     = model.predict(padded, verbose=0)[0]
    pred_idx  = np.argmax(probs)
    label     = le.classes_[pred_idx]
    confidence = probs[pred_idx] * 100

    print(f'Input      : {text}')
    print(f'Prediction : {label}')
    print(f'Confidence : {confidence:.2f}%')
    print('All scores :')
    for cls, prob in zip(le.classes_, probs):
        bar = '█' * int(prob * 30)
        print(f'  {cls:<22} {prob*100:5.1f}%  {bar}')
    print()
    return label, confidence

# Test with custom sentences
test_sentences = [
    "I feel so hopeless and empty, nothing matters anymore",
    "I am really happy today and feeling great about life",
    "I can't stop worrying about everything, my heart is racing",
    "I haven't slept in days and my thoughts are all over the place",
    "Sometimes I think everyone would be better off without me",
    "I feel overwhelmed with work and pressure from all sides"
]

print('=' * 60)
print('REAL-TIME SENTIMENT PREDICTIONS')
print('=' * 60)
for sentence in test_sentences:
    predict_sentiment(sentence, model, tokenizer, le)
    print('-' * 60)

---
## Task 9 — Save the Trained Model

In [ ]:
import os
os.makedirs('saved_models', exist_ok=True)

# 1. Save Keras model
model.save('saved_models/rnn_mental_health_model.h5')
print('✅ Model saved → saved_models/rnn_mental_health_model.h5')

# 2. Save Tokenizer
with open('saved_models/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print('✅ Tokenizer saved → saved_models/tokenizer.pkl')

# 3. Save Label Encoder
with open('saved_models/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print('✅ Label Encoder saved → saved_models/label_encoder.pkl')

# 4. Save config
config = {'vocab_size': VOCAB_SIZE, 'max_len': MAX_LEN, 'embed_dim': EMBED_DIM, 'num_classes': NUM_CLASSES}
with open('saved_models/config.pkl', 'wb') as f:
    pickle.dump(config, f)
print('✅ Config saved → saved_models/config.pkl')

print('\nAll artifacts saved successfully!')

In [ ]:
# Download saved models (Google Colab)
import shutil
shutil.make_archive('saved_models', 'zip', 'saved_models')

try:
    from google.colab import files
    files.download('saved_models.zip')
    print('Download started!')
except ImportError:
    print('Not in Colab — find saved_models.zip in your working directory')